# Session 3 — Build a U-Net
### Brain MRI Tumour Segmentation · CS Academy Seminar

---

Last session you got a Dice score out of brightness thresholding. Today you replace it with a
neural network that **learns** what a tumour looks like, and you write the architecture yourself.

By the end you will have a trained model and a number to compare against your baseline.

⏱ Roughly 2 hours. Make sure you are on a GPU: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
#@title Setup — run this first  { display-mode: "form" }
# Downloads the seminar helper code and the dataset.
REPO_RAW = "https://raw.githubusercontent.com/OTMAN-REPO/brain-mri-seminar/main"  #@param {type:"string"}
DATA_URL = ""  #@param {type:"string"}

import os, urllib.request
if not os.path.exists("seminar.py"):
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/seminar.py", "seminar.py")
        print("Got seminar.py")
    except Exception as e:
        raise SystemExit(f"Could not fetch seminar.py from {REPO_RAW}\n"
                         f"Upload it manually to this Colab session (folder icon on the left).\n{e}")

from seminar import *
import numpy as np, matplotlib.pyplot as plt
images, masks, patient_ids, slice_index = get_data(url=DATA_URL)
print(f"\n{len(images)} slices | {len(np.unique(patient_ids))} patients | image {images.shape[1:]}")
print("device:", DEVICE)

In [ ]:
import torch, torch.nn as nn, time
assert DEVICE == "cuda", "No GPU! Runtime -> Change runtime type -> T4 GPU, then re-run Setup."
print("GPU:", torch.cuda.get_device_name(0))

## 1. Why classification networks don't work here

You may have seen a network that takes an image and outputs "cat" or "dog". That network works by
throwing information away: it shrinks the image step by step until only a summary remains.

Segmentation needs the opposite. The output is **the same size as the input** — a decision for every
pixel. And here is the tension:

- To know *"is this a tumour?"* you need a wide view. Context. Which means shrinking the image.
- To know *"exactly where is the edge?"* you need fine detail. Which shrinking destroys.

The **U-Net** solves this with a trick so simple it is almost cheating:

```
 input ──► [down] ──────────────────────────────► [up] ──► output
             │                                      ▲
             └────────── skip connection ───────────┘
             ▼                                      │
           [down] ──────────────────────► [up] ─────┘
             │                              ▲
             └──────── skip connection ─────┘
             ▼                              │
                    [ bottleneck ]  ────────┘
```

Go **down** to understand *what*. Come back **up** to recover *where*. And at every step on the way
up, hand back the fine-grained feature map from the matching step on the way down — the **skip
connection**.

Without skips, a U-Net produces blurry blobs. With them, sharp edges. That is the whole idea.

## 2. The building block

Every rung of the U is the same thing twice: convolution → normalise → activate.

### Q1. Write it.

In [ ]:
class DoubleConv(nn.Module):
    """(conv 3x3 -> BatchNorm -> ReLU) twice."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        # TODO: build nn.Sequential with, in order:
        #   nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        #   nn.BatchNorm2d(out_ch)
        #   nn.ReLU(inplace=True)
        #   nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        #   nn.BatchNorm2d(out_ch)
        #   nn.ReLU(inplace=True)
        #
        # Why padding=1? So a 3x3 conv keeps the image the same size.
        # Why bias=False? BatchNorm has its own shift, so the bias would be redundant.
        self.block = ...
    def forward(self, x):
        return self.block(x)

# check
blk = DoubleConv(3, 16)
print(blk(torch.zeros(1, 3, 128, 128)).shape, "  <- want torch.Size([1, 16, 128, 128])")

## 3. The U

Four steps down, a bottleneck, four steps up. Each step down halves the image and doubles the
channels. Each step up does the reverse, then **concatenates** the skip connection before convolving.

Watch the channel counts in `c4`, `c3`, `c2`, `c1` — they take double the input channels, because
concatenation stacks the skip on top of the upsampled feature map.

### Q2. Fill in the forward pass.

In [ ]:
class MyUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, f=16):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.d1, self.d2 = DoubleConv(in_ch, f),   DoubleConv(f, f*2)
        self.d3, self.d4 = DoubleConv(f*2, f*4),   DoubleConv(f*4, f*8)
        self.bottleneck  = DoubleConv(f*8, f*16)
        self.u4, self.c4 = nn.ConvTranspose2d(f*16, f*8, 2, 2), DoubleConv(f*16, f*8)
        self.u3, self.c3 = nn.ConvTranspose2d(f*8,  f*4, 2, 2), DoubleConv(f*8,  f*4)
        self.u2, self.c2 = nn.ConvTranspose2d(f*4,  f*2, 2, 2), DoubleConv(f*4,  f*2)
        self.u1, self.c1 = nn.ConvTranspose2d(f*2,  f,   2, 2), DoubleConv(f*2,  f)
        self.out = nn.Conv2d(f, out_ch, 1)

    def forward(self, x):
        s1 = self.d1(x)                 # 128x128, keep for later
        s2 = self.d2(self.pool(s1))     #  64x64
        s3 = self.d3(self.pool(s2))     #  32x32
        s4 = self.d4(self.pool(s3))     #  16x16
        b  = self.bottleneck(self.pool(s4))   # 8x8

        # TODO: the way back up. For each level: upsample, concatenate the skip, convolve.
        # Pattern:  x = self.c4(torch.cat([self.u4(b), s4], dim=1))
        x = ...
        x = ...
        x = ...
        x = ...
        return self.out(x)              # raw logits, no sigmoid

m = MyUNet(f=16)
print("params:", sum(p.numel() for p in m.parameters())/1e6, "M")
print("output:", m(torch.zeros(2,3,128,128)).shape, " <- want torch.Size([2, 1, 128, 128])")

## 4. What are we minimising?

The network outputs a number per pixel (a *logit*). Sigmoid turns it into a probability.

Two candidate losses:

**Binary cross-entropy** — the standard per-pixel classification loss. But look at it closely and
you will see the problem from Session 2 hiding in it: it treats all pixels equally, and 99% of them
are background. BCE alone will happily drift towards predicting nothing.

**Soft Dice loss** — literally `1 - Dice`, but written differentiably (using the probability instead
of a hard 0/1 decision, so gradients can flow). It optimises the thing we actually measure.

In practice, adding them works better than either alone: BCE gives smooth, stable gradients early
on, Dice keeps the model honest about the tumour.

In [ ]:
import torch.nn.functional as F

def my_soft_dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    # TODO: per-image soft Dice, then 1 - mean.
    #   numerator   = 2 * (p * target).sum over (1,2,3)  + eps
    #   denominator = p.sum(...) + target.sum(...)       + eps
    num = ...
    den = ...
    return (1 - num/den).mean()

def my_loss(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target) + my_soft_dice_loss(logits, target)

lo = torch.randn(4,1,64,64); ta = (torch.rand(4,1,64,64) > 0.9).float()
print(f"random logits    : {my_loss(lo, ta):.4f}   (should be roughly 1-2)")
print(f"near-perfect     : {my_loss(ta*20-10, ta):.4f}   (should be close to 0)")

## 5. Train / validation split

Standard practice: hold some data back so you can measure performance on images the model has never
seen. We will shuffle all the slices and keep 20% aside.

In [ ]:
train_idx, val_idx = split_by_slice(len(images), val_frac=0.2, seed=0)
print(f"{len(train_idx)} training slices, {len(val_idx)} validation slices")

train_dl = torch.utils.data.DataLoader(
    SliceDataset(images, masks, train_idx, augment=False),
    batch_size=16, shuffle=True, num_workers=2, drop_last=True)
print(f"{len(train_dl)} batches per epoch")

## 6. Train it

In [ ]:
model = MyUNet(f=16).to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 25

history = []
for ep in range(EPOCHS):
    model.train(); tot = n = 0; t0 = time.time()
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        loss = my_loss(model(x), y)
        loss.backward(); opt.step()
        tot += loss.item(); n += 1
    val_pred = predict_all(model, images, val_idx)
    vd = dice_score(val_pred, masks[val_idx])
    history.append((tot/n, vd))
    print(f"epoch {ep+1:2d}/{EPOCHS}  loss {tot/n:.4f}  val Dice {vd:.4f}  ({time.time()-t0:.0f}s)")

torch.save(model.state_dict(), "unet_session3.pt")
print("\nsaved to unet_session3.pt")

In [ ]:
h = np.array(history)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(h[:,0]); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(h[:,1], color="seagreen"); ax[1].set_title("validation Dice"); ax[1].set_xlabel("epoch")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7. Did you beat the baseline?

In [ ]:
BASELINE_DICE = 0.30   #@param {type:"number"}   <-- put YOUR Session 2 number here

final = dice_score(predict_all(model, images, val_idx), masks[val_idx])
print(f"Session 2 threshold baseline : {BASELINE_DICE:.4f}")
print(f"Session 3 U-Net              : {final:.4f}")
print(f"\nimprovement: {final - BASELINE_DICE:+.4f}")
print(f"\nFor reference, two human radiologists agree at about 0.84 on this task.")

In [ ]:
# Look at the actual predictions
pred = predict_all(model, images, val_idx)
areas = masks[val_idx].reshape(len(val_idx), -1).sum(1)
picks = np.argsort(areas)[-8:]

fig, ax = plt.subplots(2, 8, figsize=(18, 5))
for k, p in enumerate(picks):
    j = val_idx[p]
    ax[0,k].imshow(images[j][...,1], cmap="gray")
    ax[0,k].contour(masks[j], levels=[.5], colors="lime", linewidths=1.2)
    ax[0,k].set_title("radiologist", fontsize=8)
    ax[1,k].imshow(images[j][...,1], cmap="gray")
    if pred[p].any(): ax[1,k].contour(pred[p], levels=[.5], colors="red", linewidths=1.2)
    ax[1,k].set_title(f"yours · {dice_score(pred[p], masks[j]):.2f}", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

---

## Exit ticket

1. Your final validation Dice.
2. Did you beat your Session 2 baseline? By how much?
3. How does your number compare to the ~0.84 that two radiologists achieve against each other?
4. Look at the predictions above. Where does the model do *worst*? Describe the failure in words.
5. **Keep this open.** Next session we are going to improve this model — and then find something
   quite unpleasant about the number you just wrote down.

In [ ]:
#@markdown ### Session 3 exit ticket
my_val_dice = ""  #@param {type:"string"}
beat_baseline_by = ""  #@param {type:"string"}
vs_human_agreement = ""  #@param {type:"string"}
where_it_fails = ""  #@param {type:"string"}
print("Saved. Bring unet_session3.pt to Session 4 (download it, Colab wipes files).")